<a href="https://colab.research.google.com/github/HKozuka/IDX-Exchange-Data-Science/blob/main/05_advanced_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Boosting on Random Forests

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import r2_score, mean_squared_error

# Plot styling used throughout: one series color, quiet gray ink for axes and reference lines
BLUE, INK = '#2a78d6', '#52514e'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': INK})

In [3]:
BASE_PATH = '/content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/'
TARGET = 'log_ClosePrice'
SEED = 42

df_clean = pd.read_csv(f'{BASE_PATH}X_after_feat_engin.csv')                 # training source
df_test_raw = pd.read_csv(f'{BASE_PATH}test_dataset_after_feat_eng.csv')     # holdout
print('df_clean:', df_clean.shape, '| test:', df_test_raw.shape)
df_clean.head()

df_clean: (307648, 48) | test: (12401, 47)


,Unnamed: 0,log_ClosePrice,CloseDate,ViewYN,WaterfrontYN,PoolPrivateYN,Stories,FireplaceYN,AssociationFeeFrequency,log_LivingArea,...,SqftPerBedroom,Zip_ppsf_6m,Zip_sales_6m,Nbhd_ppsf_6m,Nbhd_logprice_est,Comp_ppsf,Comp_logprice_est,Comp_size_diff,Comp_dist_km,CoastDist_km
0,0,13.610943,2024-01-05,0.0,0.0,0.0,2.0,0.0,NaN,7.588324,...,493.5,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,2.125725
1,1,13.604790,2024-01-05,0.0,0.0,0.0,2.0,0.0,NaN,7.588324,...,493.5,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,2.102132
2,2,14.557448,2024-01-02,1.0,0.0,0.0,2.0,1.0,NaN,8.226038,...,934.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,12.654789
3,3,14.665661,2024-01-31,1.0,0.0,0.0,1.0,1.0,NaN,7.800982,...,610.5,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,26.790793
4,4,13.873779,2024-01-16,1.0,0.0,0.0,1.0,1.0,NaN,7.559038,...,639.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,26.995806


## 2. Features and Time-Based Folds
Same feature matrix and expanding-window folds as `04_model_comparison.ipynb`, so every score below is directly comparable to the Random Forest's.

In [4]:
import time, shutil
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

# The training CSV was saved with its row index; the test CSV may not have it
df_clean = df_clean.drop(columns=['Unnamed: 0'], errors='ignore')
test = df_test_raw.drop(columns=['Unnamed: 0'], errors='ignore')

DROP = [TARGET, 'CloseDate', 'AssociationFee']   # AssociationFee is replaced by HOA_Monthly

def to_xy(df, feature_cols=None):
    """Numeric feature matrix and target. Pass feature_cols to match the training columns."""
    X = df.drop(columns=[c for c in DROP if c in df.columns])
    X = X.select_dtypes(include='number').astype(float).replace([np.inf, -np.inf], np.nan)
    if feature_cols is not None:
        missing = set(feature_cols) - set(X.columns)
        if missing:
            print('WARNING, missing columns:', missing)
        X = X.reindex(columns=feature_cols)
    return X, df[TARGET]

X_all, y_all = to_xy(df_clean)
FEATURES = list(X_all.columns)
X_test, y_test = to_xy(test, FEATURES)
print(f'{len(FEATURES)} features (04 used 39) | train {X_all.shape} | test {X_test.shape}')

39 features (04 used 39) | train (307648, 39) | test (12401, 39)


In [5]:
def make_key(d):
    """Address + close date + listing date, used to find each fold's rows in df_clean."""
    return (d['UnparsedAddress'].astype(str) + '|' +
            pd.to_datetime(d['CloseDate']).dt.strftime('%Y-%m-%d') + '|' +
            pd.to_datetime(d['ListingContractDate']).dt.strftime('%Y-%m-%d'))

clean_key = make_key(df_clean)
close_dates = pd.to_datetime(df_clean['CloseDate'])
cv_splits = []
for k in range(1, 6):
    fold = pd.read_csv(f'{BASE_PATH}validation_fold_{k}.csv')
    val_mask = clean_key.isin(set(make_key(fold))).values
    train_mask = (close_dates < close_dates[val_mask].min()).values & ~val_mask
    cv_splits.append((k, train_mask, val_mask))
    print(f'fold {k}: train {train_mask.sum():,}, validate {val_mask.sum():,}')
FOLD_COLS = [f'fold {k}' for k, _, _ in cv_splits]

def summarize(rows):
    """One row per setting with per-fold R2 and the mean, best first."""
    out = pd.DataFrame(rows)
    out['mean_R2'] = out[FOLD_COLS].mean(axis=1)
    return out.sort_values('mean_R2', ascending=False).reset_index(drop=True)

fold 1: train 48,810, validate 49,213
fold 2: train 98,228, validate 49,205
fold 3: train 147,305, validate 49,206
fold 4: train 196,360, validate 49,209
fold 5: train 245,586, validate 49,203


## 3. XGBoost
Light grid over `max_depth` and `learning_rate`. Each setting is fit once with 1,000 trees, then scored at 250, 500, and 1,000 trees using only the first *n* trees, so `n_estimators` costs no extra fits.

`subsample` and `colsample_bytree` play the same role as the forest's `max_samples` and `max_features`. XGBoost handles missing values on its own, so the NaNs in the neighborhood and comps features stay as they are.

In [6]:
DEVICE = 'cuda' if shutil.which('nvidia-smi') else 'cpu'   # uses a GPU runtime if one is attached
print('XGBoost device:', DEVICE)

GRID_DEPTH = [4, 6, 8]
GRID_LR = [0.05, 0.1]
GRID_N = [250, 500, 1000]

def xgb_model(depth, lr, n_trees=max(GRID_N)):
    return xgb.XGBRegressor(n_estimators=n_trees, max_depth=depth, learning_rate=lr,
                            subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                            device=DEVICE, n_jobs=-1, random_state=SEED)

rows = []
for depth in GRID_DEPTH:
    for lr in GRID_LR:
        t0 = time.time()
        scores = {n: {} for n in GRID_N}
        for k, tr, va in cv_splits:
            model = xgb_model(depth, lr).fit(X_all[tr], y_all[tr])
            for n in GRID_N:
                pred = model.predict(X_all[va], iteration_range=(0, n))
                scores[n][f'fold {k}'] = r2_score(y_all[va], pred)
        rows += [{'max_depth': depth, 'learning_rate': lr, 'n_estimators': n, **scores[n]}
                 for n in GRID_N]
        print(f'max_depth={depth}, learning_rate={lr}: {time.time() - t0:.0f}s')

xgb_results = summarize(rows)
xgb_results.head(10).style.format({c: '{:.4f}' for c in FOLD_COLS + ['mean_R2']})

XGBoost device: cuda


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [20:06:14] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


max_depth=4, learning_rate=0.05: 16s
max_depth=4, learning_rate=0.1: 15s
max_depth=6, learning_rate=0.05: 20s
max_depth=6, learning_rate=0.1: 21s
max_depth=8, learning_rate=0.05: 35s
max_depth=8, learning_rate=0.1: 36s


,max_depth,learning_rate,n_estimators,fold 1,fold 2,fold 3,fold 4,fold 5,mean_R2
0,8,0.050000,500,0.9006,0.9060,0.9119,0.9106,0.9071,0.9072
1,8,0.050000,1000,0.8984,0.9043,0.9124,0.9118,0.9083,0.9070
2,6,0.050000,1000,0.8979,0.9051,0.9104,0.9106,0.9060,0.9060
3,8,0.050000,250,0.9014,0.9045,0.9103,0.9085,0.9048,0.9059
4,6,0.050000,500,0.8999,0.9044,0.9091,0.9083,0.9041,0.9052
5,8,0.100000,250,0.8958,0.9034,0.9092,0.9094,0.9059,0.9047
6,6,0.100000,500,0.8952,0.9032,0.9099,0.9090,0.9053,0.9045
7,8,0.100000,500,0.8934,0.9017,0.9094,0.9104,0.9068,0.9043
8,6,0.100000,250,0.8965,0.9032,0.9082,0.9075,0.9033,0.9037
9,6,0.100000,1000,0.8915,0.9004,0.9101,0.9095,0.9060,0.9035


In [7]:
best_xgb = xgb_results.iloc[0]
XGB_PARAMS = dict(depth=int(best_xgb['max_depth']), lr=float(best_xgb['learning_rate']),
                  n_trees=int(best_xgb['n_estimators']))
print('Best XGBoost:', XGB_PARAMS, f"| mean validation R2 {best_xgb['mean_R2']:.4f}")

Best XGBoost: {'depth': 8, 'lr': 0.05, 'n_trees': 500} | mean validation R2 0.9072


## 4. Boosting the Random Forest's Residuals
Here XGBoost starts from the best Random Forest's prediction instead of from a constant (`base_margin`), so every tree it adds corrects what the forest gets wrong.

- **Training rows** use the forest's out-of-bag predictions: each row is predicted only by trees that never saw it. A depth-30 forest's ordinary in-sample predictions nearly memorize the target, which would leave the booster almost no error to learn from.
- **Validation rows** use the forest's normal predictions.
- The forest is refit on each fold's training window, so this section also re-scores the Random Forest on its own (it should match 04's 0.90).
- The residual booster uses shallow trees and a small learning rate, since it only has to correct, not rebuild, the forest.

In [8]:
def rf(depth=30, n_trees=100):
    """Best Random Forest from 04, with out-of-bag predictions turned on."""
    return RandomForestRegressor(n_estimators=n_trees, max_depth=depth, max_samples=0.2,
                                 max_features=0.5, oob_score=True, n_jobs=-1, random_state=SEED)

RES_DEPTH = [3, 6]
RES_LR = 0.05
RES_N = [100, 300, 600]

rf_scores = {}
res_scores = {(d, n): {} for d in RES_DEPTH for n in RES_N}
for k, tr, va in cv_splits:
    t0 = time.time()
    forest = rf().fit(X_all[tr], y_all[tr])
    rf_val = forest.predict(X_all[va])
    rf_scores[f'fold {k}'] = r2_score(y_all[va], rf_val)
    for d in RES_DEPTH:
        booster = xgb_model(d, RES_LR, max(RES_N)).fit(X_all[tr], y_all[tr],
                                                      base_margin=forest.oob_prediction_)
        for n in RES_N:
            pred = booster.predict(X_all[va], base_margin=rf_val, iteration_range=(0, n))
            res_scores[(d, n)][f'fold {k}'] = r2_score(y_all[va], pred)
    print(f'fold {k}: {time.time() - t0:.0f}s')

res_results = summarize([{'max_depth': d, 'learning_rate': RES_LR, 'n_estimators': n, **s}
                         for (d, n), s in res_scores.items()])
res_results.style.format({c: '{:.4f}' for c in FOLD_COLS + ['mean_R2']})

fold 1: 21s
fold 2: 45s
fold 3: 74s
fold 4: 99s
fold 5: 128s


,max_depth,learning_rate,n_estimators,fold 1,fold 2,fold 3,fold 4,fold 5,mean_R2
0,6,0.050000,600,0.8995,0.9059,0.9116,0.9100,0.9052,0.9064
1,6,0.050000,300,0.9004,0.9052,0.9107,0.9088,0.9039,0.9058
2,3,0.050000,600,0.8987,0.9028,0.9084,0.9066,0.9019,0.9037
3,6,0.050000,100,0.8983,0.9021,0.9082,0.9058,0.9012,0.9031
4,3,0.050000,300,0.8976,0.9017,0.9074,0.9051,0.9004,0.9024
5,3,0.050000,100,0.8943,0.8994,0.9053,0.9030,0.8983,0.9001


In [9]:
best_res = res_results.iloc[0]
RES_PARAMS = dict(depth=int(best_res['max_depth']), lr=RES_LR, n_trees=int(best_res['n_estimators']))
print('Best residual booster:', RES_PARAMS, f"| mean validation R2 {best_res['mean_R2']:.4f}")

Best residual booster: {'depth': 6, 'lr': 0.05, 'n_trees': 600} | mean validation R2 0.9064


## 5. Model Comparison
Mean validation R² across the same 5 folds. The test set is not used here.

In [10]:
comparison = pd.DataFrame({
    'Random Forest': rf_scores,
    'XGBoost': best_xgb[FOLD_COLS].astype(float).to_dict(),
    'Random Forest + XGBoost': best_res[FOLD_COLS].astype(float).to_dict(),
}).T
comparison['mean_R2'] = comparison[FOLD_COLS].mean(axis=1)
comparison['gain_vs_RF'] = comparison['mean_R2'] - comparison.loc['Random Forest', 'mean_R2']
comparison.style.format('{:.4f}')

,fold 1,fold 2,fold 3,fold 4,fold 5,mean_R2,gain_vs_RF
Random Forest,0.8901,0.8951,0.9019,0.9003,0.8954,0.8966,0.0000
XGBoost,0.9006,0.9060,0.9119,0.9106,0.9071,0.9072,0.0107
Random Forest + XGBoost,0.8995,0.9059,0.9116,0.9100,0.9052,0.9064,0.0099


## 6. Test Set Evaluation
Refit the winner on all of `df_clean`, then score the test set once. Random Forest test R² from 04 was 0.93.

In [11]:
best_name = comparison['mean_R2'].idxmax()
t0 = time.time()

if best_name == 'Random Forest':
    forest = rf().fit(X_all, y_all)
    test_pred = forest.predict(X_test)
elif best_name == 'XGBoost':
    final_xgb = xgb_model(**XGB_PARAMS).fit(X_all, y_all)
    test_pred = final_xgb.predict(X_test)
else:
    forest = rf().fit(X_all, y_all)
    final_xgb = xgb_model(**RES_PARAMS).fit(X_all, y_all, base_margin=forest.oob_prediction_)
    test_pred = final_xgb.predict(X_test, base_margin=forest.predict(X_test))

print(f'Final model: {best_name}')
print(f"Mean validation R2: {comparison.loc[best_name, 'mean_R2']:.4f}")
print(f'Test R2: {r2_score(y_test, test_pred):.4f}')
print(f'Final fit time: {time.time() - t0:.0f}s')

Final model: XGBoost
Mean validation R2: 0.9072
Test R2: 0.9461
Final fit time: 6s


In [12]:
# Accuracy in dollars, same checks as 04_model_comparison Section 8
actual = np.expm1(y_test)
predicted = np.expm1(test_pred)
abs_err = (predicted - actual).abs()

print(f'Within $10,000: {(abs_err <= 10_000).mean():.1%} of {len(actual):,} test properties')
print(f'Median absolute error: ${abs_err.median():,.0f}')
pct_err = abs_err / actual
for p in [0.05, 0.10]:
    print(f'Within {p:.0%} of sale price: {(pct_err <= p).mean():.1%}')

Within $10,000: 9.7% of 12,401 test properties
Median absolute error: $64,192
Within 5% of sale price: 37.4%
Within 10% of sale price: 63.6%


## 7. Conclusion

**XGBoost is the final model.** It beat the Random Forest on all five time-based validation folds. Its mean validation R² was 0.9072, against 0.8966 for the Random Forest. On the held-out test set, its R² was 0.946, against the Random Forest's 0.93.

**The gain is larger than the R² gap suggests.** At this level, the meaningful comparison is the variance each model leaves unexplained:

| Model | Validation R² | Test R² | Unexplained variance (test) |
|---|---|---|---|
| Random Forest | 0.8966 | 0.93 | 7.0% |
| XGBoost | 0.9072 | 0.946 | 5.4% |

XGBoost also fits on the full training set in seconds, where the forest takes minutes.

**Boosting on top of the forest did not help.** Starting XGBoost from the Random Forest's predictions scored 0.9064. That is no better than XGBoost alone, and it means training and maintaining two models, so the simpler standalone model wins.

**In dollar terms**, the final model's median absolute error is $64,192. Its predictions fall within 5% of the sale price for 37.4% of test homes and within 10% for 63.6%.